In [10]:
!pip install pyspark -q

In [11]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("CloudProviderAnalytics") \
    .getOrCreate()

# Verificamos que Spark esté activo mostrando su versión
print(f"SparkSession iniciada con éxito. Versión de Spark: {spark.version}")

SparkSession iniciada con éxito. Versión de Spark: 4.0.4


In [13]:
# Inspeccion de los archivos CSV (BATCH)
# Ruta base
LANDING_PATH = "/content/drive/MyDrive/Minería de Datos II/Proyectos/datalake/landing"

# Leemos el CSV de Organizaciones / Clientes
customers_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(f"{LANDING_PATH}/customers_orgs.csv")

print("=== ESQUEMA: customers_orgs.csv ===")
customers_df.printSchema()

print("\n=== PRIMEROS 5 REGISTROS ===")
customers_df.show(5, truncate=False)

print(f"Total de registros en clientes: {customers_df.count()}")

# Leemos el CSV de Facturación Mensual
billing_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(f"{LANDING_PATH}/billing_monthly.csv")

print("\n=== ESQUEMA: billing_monthly.csv ===")
billing_df.printSchema()

print("\n=== PRIMEROS 5 REGISTROS ===")
billing_df.show(5, truncate=False)

=== ESQUEMA: customers_orgs.csv ===
root
 |-- org_id: string (nullable = true)
 |-- org_name: string (nullable = true)
 |-- industry: string (nullable = true)
 |-- hq_region: string (nullable = true)
 |-- plan_tier: string (nullable = true)
 |-- is_enterprise: boolean (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- sales_rep: string (nullable = true)
 |-- lifecycle_stage: string (nullable = true)
 |-- marketing_source: string (nullable = true)
 |-- nps_score: double (nullable = true)


=== PRIMEROS 5 REGISTROS ===
+------------+---------------+-------------+----------+---------+-------------+-----------+---------+---------------+----------------+---------+
|org_id      |org_name       |industry     |hq_region |plan_tier|is_enterprise|signup_date|sales_rep|lifecycle_stage|marketing_source|nps_score|
+------------+---------------+-------------+----------+---------+-------------+-----------+---------+---------------+----------------+---------+
|org_xaji0y6d|Nimbus Labs 0  

Inspección de los eventos .jsonl (Streaming)

In [14]:
STREAM_PATH = f"{LANDING_PATH}/usage_events_stream/*.jsonl"

# Leemos todos los archivos JSONL de la carpeta
events_df = spark.read \
    .json(STREAM_PATH)

print("=== ESQUEMA GENERAL DE LOS EVENTOS DE USO (JSONL) ===")
events_df.printSchema()

print("\n=== MUESTRA DE EVENTOS DE USO ===")
events_df.show(5, truncate=False)

=== ESQUEMA GENERAL DE LOS EVENTOS DE USO (JSONL) ===
root
 |-- carbon_kg: double (nullable = true)
 |-- cost_usd_increment: double (nullable = true)
 |-- event_id: string (nullable = true)
 |-- genai_tokens: long (nullable = true)
 |-- metric: string (nullable = true)
 |-- org_id: string (nullable = true)
 |-- region: string (nullable = true)
 |-- resource_id: string (nullable = true)
 |-- schema_version: long (nullable = true)
 |-- service: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- unit: string (nullable = true)
 |-- value: string (nullable = true)


=== MUESTRA DE EVENTOS DE USO ===
+---------+------------------+----------------+------------+----------------+------------+----------+------------+--------------+---------+--------------------+--------+-------+
|carbon_kg|cost_usd_increment|event_id        |genai_tokens|metric          |org_id      |region    |resource_id |schema_version|service  |timestamp           |unit    |value  |
+---------+-----------

Capa Bronze (Batch)

In [15]:
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType,
    BooleanType, DateType, LongType, TimestampType
)
from pyspark.sql.functions import current_timestamp, input_file_name

# 1. Configuración de rutas principales en tu Google Drive
BASE_PATH = "/content/drive/MyDrive/Minería de Datos II/Proyectos/datalake"
LANDING_PATH = f"{BASE_PATH}/landing"
BRONZE_PATH = f"{BASE_PATH}/bronze"

print("Iniciando la ingesta a la Capa Bronze...")

# --- 2. DEFINICIÓN DE ESQUEMAS EXPLÍCITOS (STRUCTTYPE) ---

# Schema: customers_orgs
customers_schema = StructType([
    StructField("org_id", StringType(), True),
    StructField("org_name", StringType(), True),
    StructField("industry", StringType(), True),
    StructField("hq_region", StringType(), True),
    StructField("plan_tier", StringType(), True),
    StructField("is_enterprise", BooleanType(), True),
    StructField("signup_date", DateType(), True),
    StructField("sales_rep", StringType(), True),
    StructField("lifecycle_stage", StringType(), True),
    StructField("marketing_source", StringType(), True),
    StructField("nps_score", DoubleType(), True)
])

# Schema: billing_monthly
billing_schema = StructType([
    StructField("invoice_id", StringType(), True),
    StructField("org_id", StringType(), True),
    StructField("month", DateType(), True),
    StructField("subtotal", DoubleType(), True),
    StructField("credits", DoubleType(), True),
    StructField("taxes", DoubleType(), True),
    StructField("currency", StringType(), True),
    StructField("exchange_rate_to_usd", DoubleType(), True)
])

# Schema: users
users_schema = StructType([
    StructField("user_id", StringType(), True),
    StructField("org_id", StringType(), True),
    StructField("role", StringType(), True),
    StructField("created_at", TimestampType(), True),
    StructField("is_active", BooleanType(), True)
])

# Schema: resources
resources_schema = StructType([
    StructField("resource_id", StringType(), True),
    StructField("org_id", StringType(), True),
    StructField("service", StringType(), True),
    StructField("region", StringType(), True),
    StructField("created_at", TimestampType(), True),
    StructField("status", StringType(), True)
])

# Schema: support_tickets
tickets_schema = StructType([
    StructField("ticket_id", StringType(), True),
    StructField("org_id", StringType(), True),
    StructField("opened_at", TimestampType(), True),
    StructField("closed_at", TimestampType(), True),
    StructField("severity", StringType(), True),
    StructField("category", StringType(), True),
    StructField("sla_breached", BooleanType(), True),
    StructField("csat", DoubleType(), True)
])

# Schema: marketing_touches
marketing_schema = StructType([
    StructField("touch_id", StringType(), True),
    StructField("org_id", StringType(), True),
    StructField("channel", StringType(), True),
    StructField("touch_date", DateType(), True),
    StructField("converted", BooleanType(), True)
])

# Schema: nps_surveys
nps_schema = StructType([
    StructField("survey_id", StringType(), True),
    StructField("org_id", StringType(), True),
    StructField("survey_date", DateType(), True),
    StructField("score", DoubleType(), True)
])


# --- 3. PROCESAMIENTO E INGESTA BATCH A BRONZE (PARQUET) ---

datasets = [
    ("customers_orgs.csv", customers_schema, "customers_orgs"),
    ("billing_monthly.csv", billing_schema, "billing_monthly"),
    ("users.csv", users_schema, "users"),
    ("resources.csv", resources_schema, "resources"),
    ("support_tickets.csv", tickets_schema, "support_tickets"),
    ("marketing_touches.csv", marketing_schema, "marketing_touches"),
    ("nps_surveys.csv", nps_schema, "nps_surveys")
]

for file_name, schema, table_name in datasets:
    print(f"Procesando {file_name} -> Bronze ({table_name})...")

    df = spark.read \
        .option("header", "true") \
        .schema(schema) \
        .csv(f"{LANDING_PATH}/{file_name}") \
        .withColumn("ingest_ts", current_timestamp()) \
        .withColumn("source_file", input_file_name())

    # Escribimos en formato Parquet
    df.write \
        .mode("overwrite") \
        .parquet(f"{BRONZE_PATH}/{table_name}")

print("\n¡Todos los maestros se ingestaron con éxito en la Capa Bronze!")

# --- 4. VERIFICACIÓN DE LECTURA DESDE PARQUET ---

print("\n--- Verificación de la tabla Bronze 'billing_monthly' ---")
df_check = spark.read.parquet(f"{BRONZE_PATH}/billing_monthly")
df_check.printSchema()
df_check.show(3, truncate=False)

Iniciando la ingesta a la Capa Bronze...
Procesando customers_orgs.csv -> Bronze (customers_orgs)...
Procesando billing_monthly.csv -> Bronze (billing_monthly)...
Procesando users.csv -> Bronze (users)...
Procesando resources.csv -> Bronze (resources)...
Procesando support_tickets.csv -> Bronze (support_tickets)...
Procesando marketing_touches.csv -> Bronze (marketing_touches)...
Procesando nps_surveys.csv -> Bronze (nps_surveys)...

¡Todos los maestros se ingestaron con éxito en la Capa Bronze!

--- Verificación de la tabla Bronze 'billing_monthly' ---
root
 |-- invoice_id: string (nullable = true)
 |-- org_id: string (nullable = true)
 |-- month: date (nullable = true)
 |-- subtotal: double (nullable = true)
 |-- credits: double (nullable = true)
 |-- taxes: double (nullable = true)
 |-- currency: string (nullable = true)
 |-- exchange_rate_to_usd: double (nullable = true)
 |-- ingest_ts: timestamp (nullable = true)
 |-- source_file: string (nullable = true)

+------------+----------

Capa Bronze (Streaming / JSONL)

In [16]:
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, LongType
)
from pyspark.sql.functions import current_timestamp, input_file_name, to_timestamp

# 1. Esquema explícito unificado para los eventos JSONL (Soporta v1 y v2)
events_schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("timestamp", StringType(), True),
    StructField("org_id", StringType(), True),
    StructField("resource_id", StringType(), True),
    StructField("service", StringType(), True),
    StructField("region", StringType(), True),
    StructField("metric", StringType(), True),
    StructField("value", StringType(), True),             # Llega como String en Landing
    StructField("unit", StringType(), True),
    StructField("cost_usd_increment", DoubleType(), True),
    StructField("schema_version", LongType(), True),
    StructField("genai_tokens", LongType(), True),       # Presente desde v2
    StructField("carbon_kg", DoubleType(), True)          # Presente desde v2
])

CHECKPOINT_PATH = f"{BASE_PATH}/checkpoints/usage_events"

print("Iniciando Structured Streaming de usage_events_stream (*.jsonl) -> Bronze...")

# 2. Lectura en modo STREAMING desde el directorio de Landing
events_stream_df = spark.readStream \
    .schema(events_schema) \
    .json(f"{LANDING_PATH}/usage_events_stream") \
    .withColumn("ingest_ts", current_timestamp()) \
    .withColumn("source_file", input_file_name())

# 3. Escritura en STREAMING hacia Bronze con Checkpoint (Garantiza Idempotencia y Tolerancia a Fallos)
query = events_stream_df.writeStream \
    .format("parquet") \
    .outputMode("append") \
    .option("checkpointLocation", CHECKPOINT_PATH) \
    .option("path", f"{BRONZE_PATH}/usage_events") \
    .start()

# Dejamos correr el stream por unos segundos para procesar los micro-lotes existentes y luego detenemos
import time
time.sleep(10)
query.stop()

print("¡Micro-lotes procesados e ingestados con éxito en Bronze mediante Structured Streaming!")

# 4. Verificación de la lectura en Parquet
df_events_check = spark.read.parquet(f"{BRONZE_PATH}/usage_events")
print("\n--- Verificación de la tabla Bronze 'usage_events' ---")
df_events_check.printSchema()
print(f"Total de eventos ingestados en Bronze: {df_events_check.count()}")
df_events_check.show(3, truncate=False)

Iniciando Structured Streaming de usage_events_stream (*.jsonl) -> Bronze...
¡Micro-lotes procesados e ingestados con éxito en Bronze mediante Structured Streaming!

--- Verificación de la tabla Bronze 'usage_events' ---
root
 |-- event_id: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- org_id: string (nullable = true)
 |-- resource_id: string (nullable = true)
 |-- service: string (nullable = true)
 |-- region: string (nullable = true)
 |-- metric: string (nullable = true)
 |-- value: string (nullable = true)
 |-- unit: string (nullable = true)
 |-- cost_usd_increment: double (nullable = true)
 |-- schema_version: long (nullable = true)
 |-- genai_tokens: long (nullable = true)
 |-- carbon_kg: double (nullable = true)
 |-- ingest_ts: timestamp (nullable = false)
 |-- source_file: string (nullable = false)

Total de eventos ingestados en Bronze: 43200
+----------------+--------------------+------------+------------+---------+----------+----------------+-------+-